<a href="https://colab.research.google.com/github/cyruslayo/nba_bot/blob/main/notebooks/nba_market_models_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Market Models — Colab Training Notebook

Trains the `spread`, `total`, and `first_half` classification models used by `nba_bot`.

This notebook mirrors the repository training pipeline in `nba_bot/train.py` and uses the same feature builders from `nba_bot/features.py`.

Artifacts are saved to Google Drive for later use by `nba-bot-scan`.

In [ ]:
# Cell 1 — Install dependencies and force restart
import os
import sys

# Check if we've already installed
if os.path.exists('/content/.nba_bot_installed'):
    print('Dependencies already installed. Skipping to Cell 2.')
else:
    print('Installing dependencies...')
    
    # Install compatible versions
    !pip install "numpy==1.26.4" "pandas==2.2.2" "nba-on-court>=0.2.0,<1.0" xgboost scikit-learn joblib tqdm nba_api -q
    
    # Clone repo
    if not os.path.exists('/content/nba_bot'):
        !git clone https://github.com/cyruslayo/nba_bot.git /content/nba_bot
    else:
        !git -C /content/nba_bot pull
    
    !pip install -e /content/nba_bot -q
    
    # Mark as installed
    !touch /content/.nba_bot_installed
    
    print()
    print('=' * 60)
    print('⚠️  RESTARTING KERNEL...')
    print('   This will clear all imports and reload numpy')
    print('=' * 60)
    
    # Force kernel restart
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_OUTPUT = '/content/drive/MyDrive/nba_bot/'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Output dir:', DRIVE_OUTPUT)

In [ ]:
# Cell 2 — Verify imports, GPU, and XGBoost CUDA path
import json
import numpy as np
import pandas as pd
import xgboost as xgb

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('XGBoost:', xgb.__version__)

try:
    build_info = xgb.build_info()
    print('XGBoost build info:', build_info)
except Exception as e:
    print('Could not read XGBoost build info:', e)

!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv,noheader 2>/dev/null || echo 'No GPU detected'

# Small CUDA smoke test
X_smoke = np.random.rand(5000, 16).astype(np.float32)
y_smoke = (X_smoke[:, 0] + X_smoke[:, 1] * 0.5 > 0.75).astype(np.int32)

smoke_model = xgb.XGBClassifier(
    n_estimators=20,
    max_depth=4,
    learning_rate=0.1,
    tree_method='hist',
    device='cuda',
    eval_metric='logloss',
    random_state=42,
    verbosity=1,
)
smoke_model.fit(X_smoke, y_smoke, verbose=False)

config_text = smoke_model.get_booster().save_config()
config_json = json.loads(config_text)
print('Booster device entry present:', 'device' in config_text.lower())
print('CUDA smoke test complete')
print('✅ If this cell ran without fallback warnings, CUDA training is available')

In [ ]:
# Cell 3 — Training config
SEASONS = [2021, 2022, 2023, 2024]  # All seasons
MARKET_TYPES = ['spread', 'total', 'first_half']  # Processed sequentially
USE_ADVANCED_FEATURES = True
SAMPLE_GAMES_PER_SEASON = None  # Use ALL games
TEST_SIZE = 0.2
RANDOM_STATE = 42
USE_GPU = True  # Enable CUDA for XGBoost on T4

print('Seasons:', SEASONS)
print('Market types:', MARKET_TYPES)
print('Advanced features:', USE_ADVANCED_FEATURES)
print('Sample games/season:', SAMPLE_GAMES_PER_SEASON)
print('Use GPU:', USE_GPU)

In [ ]:
# Cell 4 — Load play-by-play data
import os
import pandas as pd
import nba_on_court as noc

def fix_columns(df):
    if df is None:
        return None
    df.columns = [c.upper() for c in df.columns]
    if 'GAME_ID' not in df.columns:
        possible_id_cols = [c for c in df.columns if 'NBASTATS' in c or 'GAMEID' in c or 'GAME_ID' in c]
        if possible_id_cols:
            df = df.rename(columns={possible_id_cols[0]: 'GAME_ID'})
    return df

def load_data_safe(seasons, data_type):
    try:
        df = noc.load_nba_data(seasons=seasons, data=data_type)
        if df is not None and len(df) > 0:
            return fix_columns(df)
    except Exception as e:
        print(f'Library load failed for {data_type}: {e}')

    dfs = []
    for season in seasons:
        fpath = f'/content/{data_type}_{season}.tar.xz'
        if os.path.exists(fpath):
            raw = pd.read_csv(fpath, compression='xz')
            dfs.append(fix_columns(raw))
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_nbastats = load_data_safe(SEASONS, 'nbastats')
if df_nbastats is None or df_nbastats.empty:
    raise ValueError('Could not load nbastats data')

df_pbpstats = None
if USE_ADVANCED_FEATURES:
    df_pbpstats = load_data_safe(SEASONS, 'pbpstats')

print(f'nbastats rows: {len(df_nbastats):,}')
print(f'nbastats games: {df_nbastats["GAME_ID"].nunique():,}')
if df_pbpstats is not None:
    print(f'pbpstats rows: {len(df_pbpstats):,}')
    print(f'pbpstats games: {df_pbpstats["GAME_ID"].nunique():,}')

if SAMPLE_GAMES_PER_SEASON:
    game_ids = df_nbastats['GAME_ID'].dropna().astype(str)
    season_prefix = game_ids.str[:4]
    sampled_ids = []
    for prefix in sorted(season_prefix.unique()):
        ids = sorted(df_nbastats.loc[season_prefix == prefix, 'GAME_ID'].dropna().unique().tolist())
        sampled_ids.extend(ids[:SAMPLE_GAMES_PER_SEASON])
    df_nbastats = df_nbastats[df_nbastats['GAME_ID'].isin(sampled_ids)].copy()
    if df_pbpstats is not None:
        df_pbpstats = df_pbpstats[df_pbpstats['GAME_ID'].isin(sampled_ids)].copy()
    print(f'Sampled nbastats rows: {len(df_nbastats):,}')
    print(f'Sampled games: {df_nbastats["GAME_ID"].nunique():,}')

print('✅ Data load complete')

In [ ]:
# Cell 5 — Load team ratings for T2 features
import importlib.util
import sys

player_ratings = {}
if USE_ADVANCED_FEATURES:
    !sed -i "s/per_mode_simple/per_mode_detailed/g" /content/nba_bot/nba_bot/team_stats_cache.py

    config_path = '/content/nba_bot/nba_bot/config.py'
    config_spec = importlib.util.spec_from_file_location('nba_bot.config', config_path)
    config_mod = importlib.util.module_from_spec(config_spec)
    sys.modules['nba_bot.config'] = config_mod
    config_spec.loader.exec_module(config_mod)

    module_file = '/content/nba_bot/nba_bot/team_stats_cache.py'
    spec = importlib.util.spec_from_file_location('nba_bot.team_stats_cache', module_file)
    tsc_module = importlib.util.module_from_spec(spec)
    sys.modules['nba_bot.team_stats_cache'] = tsc_module
    spec.loader.exec_module(tsc_module)

    refresh_team_stats = tsc_module.refresh_team_stats
    player_ratings = refresh_team_stats(output_path='/content/team_stats.json')
    print(f'Loaded ratings for {len(player_ratings)} teams')
else:
    print('Skipping team ratings')

In [ ]:
# Cell 6 — Training helpers (shared by the market-specific cells)
import gc
import json
import os
import joblib
import numpy as np
import xgboost as xgb
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
import importlib.util
import sys

features_path = '/content/nba_bot/nba_bot/features.py'
spec = importlib.util.spec_from_file_location('nba_bot.features', features_path)
features_module = importlib.util.module_from_spec(spec)
sys.modules['nba_bot.features'] = features_module
spec.loader.exec_module(features_module)

build_spread_rows = features_module.build_spread_rows
build_total_rows = features_module.build_total_rows
build_first_half_rows = features_module.build_first_half_rows

from nba_bot.config import FEATURES_SPREAD, FEATURES_TOTAL, FEATURES_FIRST_HALF

results = {}

def train_market_model(df_market, feature_cols, target_col, model_name, feature_name, market_label):
    print('=' * 70)
    print(f'Training {market_label}')
    print('=' * 70)
    print(f'Raw dataset: {df_market.shape[0]:,} rows, {df_market.shape[1]} cols')

    df_market = df_market[df_market['time_remaining'] < 2870].copy()
    df_market = df_market.dropna(subset=feature_cols + [target_col])

    print(f'Filtered rows: {len(df_market):,}')
    print(f'Target mean: {df_market[target_col].mean():.4f}')

    X = df_market[feature_cols].to_numpy(dtype=np.float32, copy=True)
    y = df_market[target_col].to_numpy(copy=True)

    del df_market
    gc.collect()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    del X, y
    gc.collect()

    print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        device='cuda' if USE_GPU else 'cpu',
        predictor='auto',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        verbosity=1,
    )

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

    booster_config = json.loads(model.get_booster().save_config())
    print('Booster config loaded')
    print('Requested device:', 'cuda' if USE_GPU else 'cpu')

    probs = model.predict_proba(X_test)[:, 1]
    metrics = {
        'log_loss': round(float(log_loss(y_test, probs)), 4),
        'brier_score': round(float(brier_score_loss(y_test, probs)), 4),
        'auc_roc': round(float(roc_auc_score(y_test, probs)), 4),
    }
    print('Metrics:', metrics)

    model_path = os.path.join(DRIVE_OUTPUT, model_name)
    feature_path = os.path.join(DRIVE_OUTPUT, feature_name)
    joblib.dump(model, model_path)
    joblib.dump(feature_cols, feature_path)

    results[market_label] = {
        'metrics': metrics,
        'model_path': model_path,
        'feature_path': feature_path,
    }

    print('Saved model:', model_path)
    print('Saved features:', feature_path)

    del model, X_train, X_test, y_train, y_test, probs, booster_config
    gc.collect()
    print('Memory cleared')
    return results[market_label]

print('✅ Helper functions loaded')

In [ ]:
# Cell 7 — Train spread model
if 'spread' in MARKET_TYPES:
    df_spread = build_spread_rows(
        df_nbastats=df_nbastats,
        df_pbpstats=df_pbpstats,
        player_ratings=player_ratings,
    )
    spread_result = train_market_model(
        df_market=df_spread,
        feature_cols=FEATURES_SPREAD,
        target_col='covered_spread',
        model_name='xgb_spread_t2.pkl',
        feature_name='feature_cols_spread.pkl',
        market_label='spread',
    )
    del df_spread
    gc.collect()
else:
    print('Skipping spread')

In [ ]:
# Cell 8 — Train total model
if 'total' in MARKET_TYPES:
    df_total = build_total_rows(
        df_nbastats=df_nbastats,
        df_pbpstats=df_pbpstats,
        player_ratings=player_ratings,
    )
    total_result = train_market_model(
        df_market=df_total,
        feature_cols=FEATURES_TOTAL,
        target_col='went_over',
        model_name='xgb_total_t2.pkl',
        feature_name='feature_cols_total.pkl',
        market_label='total',
    )
    del df_total
    gc.collect()
else:
    print('Skipping total')

In [ ]:
# Cell 9 — Train first-half model
if 'first_half' in MARKET_TYPES:
    df_first_half = build_first_half_rows(
        df_nbastats=df_nbastats,
        df_pbpstats=df_pbpstats,
        player_ratings=player_ratings,
    )
    first_half_result = train_market_model(
        df_market=df_first_half,
        feature_cols=FEATURES_FIRST_HALF,
        target_col='home_win',
        model_name='xgb_first_half_t2.pkl',
        feature_name='feature_cols_first_half.pkl',
        market_label='first_half',
    )
    del df_first_half
    gc.collect()
else:
    print('Skipping first_half')

In [ ]:
# Cell 10 — Export summary
print('Completed markets:', list(results.keys()))
print()
for market_type, info in results.items():
    print('=' * 70)
    print(market_type)
    print('metrics:', info['metrics'])
    print('model:', info['model_path'])
    print('features:', info['feature_path'])

print('')
print('Set these locally after downloading the artifacts:')
print('  export NBA_BOT_SPREAD_MODEL_PATH=/path/to/xgb_spread_t2.pkl')
print('  export NBA_BOT_TOTAL_MODEL_PATH=/path/to/xgb_total_t2.pkl')
print('  export NBA_BOT_FIRST_HALF_MODEL_PATH=/path/to/xgb_first_half_t2.pkl')
print('  export NBA_BOT_ENABLE_SPREAD_TRADING=true')
print('  export NBA_BOT_ENABLE_TOTAL_TRADING=true')
print('  export NBA_BOT_ENABLE_FIRST_HALF_TRADING=true')
print('')
print('Run one market cell at a time if you want to watch memory/GPU behavior closely.')